In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file("key.json") # lokalizacja pobranego klucza z punktu 1.4.

client = bigquery.Client(credentials=credentials, project=credentials.project_id) 

In [ ]:
gsod:pd.DataFrame
if False:
    query = """
    SELECT *
    FROM `bigquery-public-data.noaa_gsod.gsod*` WHERE _TABLE_SUFFIX BETWEEN '2020' AND '2023'
    """

    query_job = client.query(query)
    query_result = query_job.result()
    gsod = query_result.to_dataframe()
    gsod.to_csv("gsod2020.csv.gz", index=False)
else:
    gsod = pd.read_csv("gsod2020.csv.gz")


In [ ]:
gsod

In [ ]:
isd = pd.read_csv("https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv")
isd.head()

In [ ]:
isd["WBAN"] = isd["WBAN"].apply(lambda x: str(x).zfill(5))
gsod["wban"] = gsod["wban"].apply(lambda x: str(x).zfill(5))
df = gsod.merge(isd, how="left", left_on=["stn", "wban"], right_on=["USAF", "WBAN"])

Czyszczenie danych z nieprawidłowych wartości

In [ ]:
df["temp"] = df["temp"].replace(9999.9, np.nan)
df["dewp"] = df["dewp"].replace(9999.9, np.nan)
df["slp"] = df["slp"].replace(9999.9, np.nan)
df["stp"] = df["stp"].replace(9999.9, np.nan) # podejrzana max wartość 999.9 (ale to blisko 1 bar)
df["visib"] = df["visib"].replace(999.9, np.nan)
df["wdsp"] = df["wdsp"].replace(999.9, np.nan)
df["mxpsd"] = df["mxpsd"].replace(999.9, np.nan)
df["gust"] = df["gust"].replace(999.9, np.nan)
df["max"] = df["max"].replace(9999.9, np.nan)
df["min"] = df["min"].replace(9999.9, np.nan)
df["prcp"] = df["prcp"].replace(99.99, np.nan)
df["sndp"] = df["sndp"].replace(999.9, np.nan)

In [ ]:
df["fog"] = df["fog"].astype('bool')
df["rain_drizzle"] = df["rain_drizzle"].astype('bool')
df["snow_ice_pellets"] = df["snow_ice_pellets"].astype('bool')
df["hail"] = df["hail"].astype('bool')
df["thunder"] = df["thunder"].astype('bool')
df["tornado_funnel_cloud"] = df["tornado_funnel_cloud"].astype('bool')

df["date"] = pd.to_datetime(df["date"])

In [ ]:
df.describe().T

3.1 Ilość wierszy z danymi

In [ ]:
len(df)

3.2 Ilość krajów

In [ ]:
df["CTRY"].unique()

In [ ]:
len(df["CTRY"].unique())

3.3. W jaki sposób zapisywane są dzienne informacje dla poszczególnych lokalizacji.

In [ ]:
df[["stn", "wban", "date", "year", "mo", "da"]].head()

3.4. Jak zapisane są wartości liczbowe

Chyba stopnie Fahrenheita

In [ ]:
df["temp"]

3.5 Przedział czasowy

In [ ]:
df["date"].describe().T

In [ ]:
df[(df["prcp"].notna()) & (df["date"] != 0)]["date"].describe().T

In [ ]:
df[(df["temp"].notna())]["date"].describe().T

3.6 Więcej informacji o danych pogodowych.

In [ ]:
df[["dewp", "temp", "fog", "wdsp", "snow_ice_pellets"]]

4.1. Podstawowe informacje o lokalizacjach pomiarów pogodowych (stacje) oraz krajach, tak aby dane były zrozumiałe dla człowieka i możliwe do dalszego przetwarzania.

In [ ]:
country_codes = pd.read_csv("https://raw.githubusercontent.com/mysociety/gaze/refs/heads/master/data/fips-10-4-to-iso-country-codes.csv")
country_codes

In [ ]:
stations = df[["stn", "wban", "STATION NAME", "CTRY", "STATE", "LAT", "LON"]].drop_duplicates()
stations = stations[stations["CTRY"].notna()]

stations_merged = stations.merge(
    country_codes, 
    how="left", 
    left_on="CTRY", 
    right_on="FIPS 10-4"
)
stations_merged.rename(columns={"Name": "Country", "STATION NAME": "Station Name", "STATE": "State", "LAT": "lat", "LON": "lon"}, inplace=True)
stations_merged[["stn", "wban", "Station Name", "Country", "State", "lat", "lon", "FIPS 10-4", "ISO 3166"]]
stations_merged.to_csv("stations.csv", index=False)

4.2. Chcemy wygenerować podstawowe zestawienia dotyczące warunków pogodowych na świecie (np. temperatura, opady, wiatr) w ujęciu dziennym.

In [ ]:
simplified_df = df.sort_values("date")[["date", "stn", "wban", "temp", "prcp", "wdsp"]]

In [ ]:
simplified_df.to_csv("weather.csv.gz", index=False)

4.3. Chcemy poznać zjawiska ekstremalne w danych pogodowych poprzez uwypuklenie skrajnych wartości (np. bardzo wysokie/niskie temperatury, intensywne opady, silny wiatr).

In [ ]:
temp_low, temp_high = simplified_df['temp'].quantile([0.01, 0.99])
prcp_high = simplified_df['prcp'].quantile(0.99)
wdsp_high = simplified_df['wdsp'].quantile(0.99)

# filtrowanie ekstremów
extreme_records = simplified_df[
    (simplified_df['temp'] < temp_low) | (simplified_df['temp'] > temp_high) |
    (simplified_df['prcp'] > prcp_high) |
    (simplified_df['wdsp'] > wdsp_high)
]
extreme_records.to_csv("extreme_weather.csv.gz", index=False)

In [ ]:
extreme_records

4.4. Chcemy przygotować dane umożliwiające analizę zmian warunków pogodowych w czasie (np. dla wybranych lokalizacji lub zmiennych).

In [ ]:
lask_station = stations_merged[stations_merged["Station Name"] == "LASK"].reset_index().T[0]
lask_weather = simplified_df[(simplified_df["stn"] == lask_station["stn"]) & (simplified_df["wban"] == lask_station["wban"])]
lask_weather.to_csv("lask_weather.csv.gz", index=False)

In [ ]:
lask_weather.plot(x="date", y="temp", title="Temperature over time in LASK station", ylabel="Temperature (°F)", xlabel="Date")

4.5. Zdefiniuj własny dodatkowy przypadek związany z danymi pogodowymi.

In [ ]:
custom_weather = simplified_df[(simplified_df["temp"] < 32) & (simplified_df["prcp"] > 0)]
custom_weather.to_csv("custom_weather.csv.gz", index=False)
custom_weather

5. Finalny plik csv to weather.csv.gz ponieważ w każdym punkcie zadania czwartego filtrujemy z simplified_df

6. Połącz ze sobą dane otrzymane w części 5 oraz dane znajdujące się w dodatkowym pliku CSV zawierającym informacje o produkcji rolniczej.

In [ ]:
m49_to_iso = pd.read_csv("https://econweb.ucsd.edu/muendler/teach/20f/435/conc/unsd-country-codes.csv")[["M49 Code", "ISO-alpha2 Code"]]
m49_to_iso

In [ ]:
faostat = pd.read_csv("FAOSTAT_data_en_3-16-2026.csv")
merged_faostat = faostat.merge(m49_to_iso, how="left", left_on="Area Code (M49)", right_on="M49 Code")
merged_faostat
# faostat